In [5]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Grille 120x120 (rapide et fluide)
N = 120
x, y = np.meshgrid(np.linspace(-1, 1, N), np.linspace(-1, 1, N))

# Masque cérébral (ellipse)
brain_mask = (x**2 / 0.85**2 + y**2 / 0.95**2) <= 1.0

# Substance grise corticale (anneau périphérique)
cortex_mask = brain_mask & ((x**2 / 0.75**2 + y**2 / 0.85**2) > 1.0)

# Substance blanche centrale
wm_mask = brain_mask & ~cortex_mask

# Ventricules latéraux (papillon central -> LCR, MWF = 0%)
vent_left = ((x + 0.16)**2 / 0.08**2 + (y - 0.05)**2 / 0.28**2) <= 1.0
vent_right = ((x - 0.16)**2 / 0.08**2 + (y - 0.05)**2 / 0.28**2) <= 1.0
ventricles = vent_left | vent_right

# Corps calleux (genou et splénium -> très riche en myéline, MWF ~ 22-25%)
genu = (x**2 / 0.22**2 + (y - 0.36)**2 / 0.07**2) <= 1.0
splenium = (x**2 / 0.26**2 + (y + 0.36)**2 / 0.08**2) <= 1.0
corpus_callosum = (genu | splenium) & ~ventricles

# Construction de la carte MWF saine (%)
rng = np.random.default_rng(42)
mwf_healthy = np.full((N, N), np.nan)
mwf_healthy[cortex_mask] = 4.5 + rng.normal(0, 0.5, cortex_mask.sum())
mwf_healthy[wm_mask] = 15.0 + rng.normal(0, 1.0, wm_mask.sum())
mwf_healthy[corpus_callosum] = 23.0 + rng.normal(0, 1.0, corpus_callosum.sum())
mwf_healthy[ventricles] = 0.0
mwf_healthy = np.clip(mwf_healthy, 0, 30)

# Construction de la carte avec lésion démyélinisante (ex: SEP périventriculaire)
lesion_mask = ((x + 0.35)**2 / 0.11**2 + (y - 0.12)**2 / 0.11**2) <= 1.0
mwf_lesion = mwf_healthy.copy()
mwf_lesion[lesion_mask] = 3.5 + rng.normal(0, 0.5, lesion_mask.sum())

# Image pondérée T2 qualitative correspondante (unités arbitraires)
t2w_lesion = np.full((N, N), np.nan)
t2w_lesion[cortex_mask] = 110
t2w_lesion[wm_mask] = 75
t2w_lesion[corpus_callosum] = 65
t2w_lesion[ventricles] = 220
t2w_lesion[lesion_mask] = 175  # Hyperintense en T2w

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Qualitative T2-weighted MRI (a.u.)", "Quantitative MWF Map (%)")
)

# Sous-graphe 1 : T2w
fig.add_trace(
    go.Heatmap(
        z=t2w_lesion,
        colorscale="Gray",
        showscale=False,
        hovertemplate="X: %{x}<br>Y: %{y}<br>T2w signal: %{z:.0f}<extra></extra>"
    ),
    row=1, col=1
)

# Sous-graphe 2 : MWF sain (visible par défaut)
fig.add_trace(
    go.Heatmap(
        z=mwf_healthy,
        colorscale="Viridis",
        zmin=0, zmax=28,
        colorbar=dict(title="MWF (%)"),
        hovertemplate="X: %{x}<br>Y: %{y}<br>MWF: %{z:.1f}%<extra></extra>",
        visible=True
    ),
    row=1, col=2
)

# Sous-graphe 2 : MWF avec lésion (masqué par défaut)
fig.add_trace(
    go.Heatmap(
        z=mwf_lesion,
        colorscale="Viridis",
        zmin=0, zmax=28,
        colorbar=dict(title="MWF (%)"),
        hovertemplate="X: %{x}<br>Y: %{y}<br>MWF: %{z:.1f}%<extra></extra>",
        visible=False
    ),
    row=1, col=2
)

fig.update_layout(
    title=dict(
        text="Brain Axial Slice: T2-Weighted Contrast vs. Quantitative MWF Map",
        y=0.96,
        x=0.5,
        xanchor="center"
    ),
    template="plotly_white",
    height=560,
    width=850,
    margin=dict(t=130, b=40, l=40, r=40),
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            x=0.5,
            y=1.13,
            xanchor="center",
            yanchor="top",
            buttons=[
                dict(
                    label="Healthy White Matter",
                    method="update",
                    args=[{"visible": [True, True, False]}]
                ),
                dict(
                    label="Demyelinating Lesion (MS)",
                    method="update",
                    args=[{"visible": [True, False, True]}]
                )
            ]
        )
    ]
)

fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(showticklabels=False, scaleanchor="x", scaleratio=1, row=1, col=1)
fig.update_xaxes(showticklabels=False, row=1, col=2)
fig.update_yaxes(showticklabels=False, scaleanchor="x2", scaleratio=1, row=1, col=2)

fig.show()